# Monocle3 Trajectory Analysis

In [ ]:
# !pip install --quiet scanpy leidenalg igraph statsmodels scikit-learn
# !pip install --quiet git+https://github.com/bioturing/py-monocle.git@a0d77b63309af6939cb0a8799471e543fa328ac8
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/tumour/pulmonary_epi_Lin/

In [ ]:
import os
import warnings

import matplotlib as mpl
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from matplotlib.colors import ListedColormap
from py_monocle import learn_graph, order_cells
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.stats import entropy, gaussian_kde, rankdata, spearmanr
from scipy.stats import t as t_dist
from sklearn.preprocessing import MinMaxScaler
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)

## Config

In [ ]:
input_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join_cytotrace.h5ad"

output_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join_cytotrace_monocle.h5ad"

malignant_output_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join_cytotrace_monocle_malignant.h5ad"
figures_output_dir = "./figures/monocle"
tables_output_dir = "./tables"
seed = 0

state_col = "final_state"
base_label_col = "cell_type_new"
malignant_label = "Malignant"
donor_col = "donor"
counts_layer = "raw_counts"
ct_score_col = "ct_score"               

# --- trajectory ---
embedding_basis = "X_draw_graph_fa"     # FDG of scVI (HVG) latent space, computed upstream
root_state = "Alveolar cell type II"     # root = AT2 cell closest to the AT2 centroid
pseudotime_col = "monocle_pseudotime"
pseudotime_scaled_col = "pseudotime_scaled"

# display order for ridge plots (also defines the states included in the trajectory)
state_order = [
    "Alveolar cell type I", "Alveolar cell type II", "Transitional",
    "1_Epithelial baseline", "4_Inflammatory EMT", "2_Differentiated epithelial",
    "3_Devp_stem-like", "5_Neurod_EMT-like", "6_Proliferative",
]
state_colors = {
    "Alveolar cell type I": "#7d87b9",
    "Alveolar cell type II": "#bec1d4",
    "Transitional": "#63ffac",
    "1_Epithelial baseline": "#8dd3c7",
    "2_Differentiated epithelial": "#bebada",
    "3_Devp_stem-like": "#80b1d3",
    "4_Inflammatory EMT": "#fccde5",
    "5_Neurod_EMT-like": "#bc80bd",
    "6_Proliferative": "#ffed6f",
}

recist_order = ["0_adj_normal", "1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
treatment_order = ["1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
recist_colors = {
    "0_adj_normal": "#d86464", "1_treatment_naive": "#f4c9c9",
    "3_on_treatment": "#af8b8b", "4_TKI_resistance": "#636262",
}

# --- ridge / scatter plots ---
kde_bandwidth = 0.3
kde_min_cells = 10
lowess_frac = 0.25

# --- PAGA on the malignant compartment ---
paga_use_rep = "X_scVI_n_latent_hvg_10"
paga_n_neighbors = 40
paga_kwargs = dict(threshold=0.1, node_size_scale=2.0, node_size_power=0.5,
                   edge_width_scale=0.4, fontsize=6, show=False)
pseudotime_cmap = "GnBu"
ct_score_cmap = "RdYlBu_r"

# --- response groups (on-treatment patients split by best response) ---
response_groups = {
    "Naive": ["TN001", "TN002", "TN003", "TN004", "TN005", "TN006", "TN007", "TN008", "TN009"],
    "PR-1": ["P012", "P009"],
    "PR-2": ["P011", "P003", "P001", "P005"],
    "SD": ["P006", "P008", "P002", "P013"],
    "Resistant": ["R006", "R009", "R005", "R010", "R007", "R003", "R001"],
}
response_order = ["Naive", "PR-1", "PR-2", "SD", "Resistant"]
response_colors = {
    "Naive": "#f4c9c9", "PR-1": "#55A868", "PR-2": "#8172B3",
    "SD": "#C44E52", "Resistant": "#636262",
}
# malignant cells from adjacent-normal samples share the TN donor ID; [] reproduces the original grouping
response_exclude_recist = []

# --- donor-level plasticity / resistance scores ---
resistant_states = ["4_Inflammatory EMT", "5_Neurod_EMT-like", "6_Proliferative"]
potential_features = ["plasticity_mean", "entropy"]
commitment_features = ["pseudotime_mean", "pseudotime_max", "resistance_fraction", "entropy_inverted"]

# --- pseudotime-associated genes and heatmaps ---
exclude_gene_regex = r"^(RPS|RPL|MT-|LINC)|^(AC|AL|AP)\d+"
min_gene_detection = 0.05        # fraction of malignant cells expressing the gene
gene_test_max_cells = 20000      # cells subsampled for the gene-pseudotime Spearman test
gene_padj_threshold = 0.05
n_top_genes = 200
heatmap_max_cells = 5000         # cells subsampled for heatmap display
baseline_state = "1_Epithelial baseline"
path2_states = ["1_Epithelial baseline", "4_Inflammatory EMT", "5_Neurod_EMT-like", "6_Proliferative"]

# obs columns kept in the malignant output
keep_obs = ["sample", donor_col, "ID", state_col, "recist_v3", pseudotime_col,
            pseudotime_scaled_col, ct_score_col, "ct_pseudotime", "response_group"]

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
rng = np.random.default_rng(seed)

## 1. Load data and subset to trajectory states

In [ ]:
adata = sc.read(input_path)
adata = adata[adata.obs[state_col].isin(state_order)].copy()
adata.obs[state_col] = pd.Categorical(adata.obs[state_col], categories=state_order)
adata.uns[f"{state_col}_colors"] = [state_colors[s] for s in state_order]
adata.uns.pop("iroot", None)

adata.obs[state_col].value_counts()

## 2. Learn principal graph and order cells

Graph is learned on the 2D FDG layout with `final_state` as the partition, so
branch structure is constrained by the state annotation. Pseudotime is in FDG
distance units (~10⁵); the 0-1 min-max scaled version is used for all plots.

In [ ]:
embedding = adata.obsm[embedding_basis]
clusters = adata.obs[state_col].cat.codes.values

projected_points, mst, centroids = learn_graph(matrix=embedding, clusters=clusters)

root_mask = (adata.obs[state_col] == root_state).values
root_indices = np.where(root_mask)[0]
root_centroid = embedding[root_mask].mean(axis=0)
root_cell_idx = root_indices[np.argmin(np.linalg.norm(embedding[root_mask] - root_centroid, axis=1))]
print(f"root cell: {adata.obs_names[root_cell_idx]} (index {root_cell_idx})")

pseudotime = order_cells(embedding, centroids, mst=mst, projected_points=projected_points,
                         root_cells=np.array([root_cell_idx]))
adata.obs[pseudotime_col] = pseudotime
pt = adata.obs[pseudotime_col]
adata.obs[pseudotime_scaled_col] = (pt - pt.min()) / (pt.max() - pt.min())

adata.obs.groupby(state_col, observed=True)[pseudotime_scaled_col].describe()

## 3. Trajectory on the FDG layout

In [ ]:
edges = np.array(mst.nonzero()).T
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

sc.pl.embedding(adata, basis=embedding_basis, color=state_col, ax=axes[0], show=False,
                title="A. Cell states", legend_loc="on data", frameon=False)

scatter = axes[1].scatter(embedding[:, 0], embedding[:, 1], c=adata.obs[pseudotime_scaled_col],
                          s=4, cmap=pseudotime_cmap, vmin=0, vmax=1, rasterized=True)
for edge in edges:
    axes[1].plot(centroids[edge, 0], centroids[edge, 1], c="red", linewidth=0.5, alpha=0.9, zorder=10)
axes[1].scatter(*embedding[root_cell_idx], c="red", s=150, marker="*", edgecolors="black",
                linewidths=1, zorder=11, label=f"Root ({root_state})")
axes[1].set_title("B. Pseudotime trajectory", fontsize=14)
axes[1].legend(loc="lower right", frameon=False)
axes[1].set_xticks([]); axes[1].set_yticks([])
for spine in axes[1].spines.values():
    spine.set_visible(False)
plt.colorbar(scatter, ax=axes[1], label="Scaled pseudotime (0–1)", fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(f"{figures_output_dir}/01_final_state_trajectory_pseudotime.pdf", dpi=150, bbox_inches="tight")
plt.show()

## 4. Pseudotime distribution by cell state

One reusable function replaces the original per-group ridge cells. All panels use
the globally scaled pseudotime, so positions are comparable across RECIST groups.

In [ ]:
def plot_ridge(obs_df, value_col, states, ax, colors=state_colors,
               min_cells=kde_min_cells, bw=kde_bandwidth):
    """Normalised KDE ridge per state on a 0–1 axis; states with < min_cells are left empty."""
    grid = np.linspace(0, 1, 200)
    for i, state in enumerate(states):
        vals = obs_df.loc[obs_df[state_col] == state, value_col].dropna().values
        if len(vals) < min_cells or np.std(vals) == 0:
            continue
        density = gaussian_kde(vals, bw_method=bw)(grid)
        ax.fill_between(grid, i, i + density / density.max() * 0.9,
                        color=colors[state], alpha=0.9, linewidth=0)
    ax.set_xlim(0, 1)
    ax.set_yticks(np.arange(len(states)))
    ax.set_xlabel("Scaled pseudotime (0–1)")
    sns.despine(ax=ax)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
plot_ridge(adata.obs, pseudotime_scaled_col, state_order, ax)
ax.set_yticklabels(state_order, fontsize=10)
ax.set_title("Trajectory progression")
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/02_pseudotime_ridge_by_state.pdf", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, len(recist_order), figsize=(4 * len(recist_order), 3), sharey=True)
for ax, recist in zip(axes, recist_order):
    plot_ridge(adata.obs[adata.obs["recist_v3"] == recist], pseudotime_scaled_col, state_order, ax)
    ax.set_title(recist)
axes[0].set_yticklabels(state_order, fontsize=10)
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/03_pseudotime_ridge_by_state_recist.pdf", bbox_inches="tight")
plt.show()

## 5. CytoTRACE vs pseudotime

Cell-level Spearman p-values are reported for completeness but are not
informative at n ≈ 10⁴ cells; the patient-level correlation (one median pair per
donor) is the one to interpret.

In [ ]:
def raster_scatter_lowess(data, x, y, hue, palette, **kwargs):
    ax = plt.gca()
    for state, subset in data.groupby(hue, observed=True):
        ax.scatter(subset[x], subset[y], s=6, alpha=0.8, color=palette[state], rasterized=True)
    d = data[[x, y]].dropna().sort_values(x)
    smoothed = lowess(d[y], d[x], frac=lowess_frac, delta=0.01 * np.ptp(d[x]))
    ax.plot(smoothed[:, 0], smoothed[:, 1], color="black", linewidth=0.5, zorder=10)

g = sns.FacetGrid(adata.obs, col="recist_v3", col_order=recist_order,
                  height=4, aspect=0.8, sharex=True, sharey=True)
g.map_dataframe(raster_scatter_lowess, x=pseudotime_scaled_col, y=ct_score_col,
                hue=state_col, palette=state_colors)
g.set_axis_labels("Scaled pseudotime (0–1)", "CytoTRACE score")
g.set_titles("{col_name}")
handles = [mlines.Line2D([], [], color=state_colors[s], marker="o", linestyle="None", markersize=6, label=s)
           for s in state_order]
plt.legend(handles=handles, title="State", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/04_ctscore_vs_pseudotime_by_recist.pdf", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
rows = []
for recist in recist_order:
    d = adata.obs.loc[adata.obs["recist_v3"] == recist, [donor_col, pseudotime_scaled_col, ct_score_col]].dropna()
    rho_cell, p_cell = spearmanr(d[pseudotime_scaled_col], d[ct_score_col])
    per_donor = d.groupby(donor_col, observed=True)[[pseudotime_scaled_col, ct_score_col]].median()
    rho_donor, p_donor = (spearmanr(per_donor[pseudotime_scaled_col], per_donor[ct_score_col])
                          if len(per_donor) >= 3 else (np.nan, np.nan))
    rows.append({"recist_v3": recist, "n_cells": len(d), "rho_cell": rho_cell, "p_cell": p_cell,
                 "n_donors": len(per_donor), "rho_donor_median": rho_donor, "p_donor_median": p_donor})

corr_df = pd.DataFrame(rows)
corr_df.to_csv(f"{tables_output_dir}/monocle_ctscore_pseudotime_spearman.csv", index=False)
corr_df

## 6. Save trajectory output

`X` is restored to raw counts. A reduced malignant-only object is also written for
sections 7–9.

In [ ]:
adata.X = adata.layers[counts_layer].copy()
adata.uns.pop("log1p", None)
adata.write(output_path, compression="gzip")

donor_to_group = {d: g for g, donors in response_groups.items() for d in donors}
adata_mal = adata[adata.obs[base_label_col] == malignant_label].copy()
adata_mal.obs["response_group"] = adata_mal.obs[donor_col].astype(str).map(donor_to_group)
adata_mal.obs = adata_mal.obs[keep_obs]
adata_mal.obsp.clear()
adata_mal.write(malignant_output_path, compression="gzip")

print(adata_mal.obs[state_col].value_counts(), "\n")
print(adata_mal.obs["response_group"].value_counts(dropna=False))

## 7. PAGA abstraction of the malignant compartment

Neighbours and PAGA are recomputed within each group. Node colour is the mean
value per state; all panels in a row share one colour scale, so the colour bars
are valid across groups.

In [ ]:
def plot_paga_grid(ad, group_col=None, groups=None, save_path=None, figsize=(6, 2.2)):
    """Rows: cell states / scaled pseudotime / CytoTRACE; one column per group (or one row if no groups)."""
    groups = groups or [None]
    subsets = {}
    for grp in groups:
        sub = ad if grp is None else ad[ad.obs[group_col] == grp].copy()
        sc.pp.neighbors(sub, use_rep=paga_use_rep, n_neighbors=paga_n_neighbors, key_added="paga_neighbors")
        sc.tl.paga(sub, groups=state_col, neighbors_key="paga_neighbors")
        sub.uns.pop("iroot", None)
        subsets[grp] = sub

    rows = [(pseudotime_scaled_col, pseudotime_cmap, "Pseudotime"), (ct_score_col, ct_score_cmap, "CytoTRACE")]
    limits = {}
    for col, _, _ in rows:
        means = pd.concat([s.obs.groupby(state_col, observed=True)[col].mean() for s in subsets.values()])
        limits[col] = (means.min(), means.max())

    single = len(groups) == 1
    fig, axes = plt.subplots(1, 3, figsize=figsize) if single else plt.subplots(3, len(groups), figsize=figsize)
    get_ax = (lambda r, c: axes[r]) if single else (lambda r, c: axes[r, c])

    for c, (grp, sub) in enumerate(subsets.items()):
        title = "Cell states" if grp is None else str(grp).replace("_", " ")
        sc.pl.paga(sub, color=state_col, ax=get_ax(0, c), title=title, **paga_kwargs)
        for r, (col, cmap, label) in enumerate(rows, start=1):
            ax = get_ax(r, c)
            sc.pl.paga(sub, color=col, cmap=cmap, colorbar=False, ax=ax,
                       title=label if single else "", **paga_kwargs)
            for coll in ax.collections:
                if coll.get_array() is not None:
                    coll.set_clim(*limits[col])

    for ax in np.ravel(axes):
        x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
        ax.set_xlim(x0 - 0.1 * (x1 - x0), x1 + 0.1 * (x1 - x0))
        ax.set_ylim(y0 - 0.1 * (y1 - y0), y1 + 0.1 * (y1 - y0))
        for spine in ax.spines.values():
            spine.set_visible(False)
    if not single:
        for r, label in enumerate(["Cell states", "Pseudotime", "CytoTRACE"]):
            axes[r, 0].set_ylabel(label, fontsize=7)

    for (col, cmap, label), bottom in zip(rows, (0.55, 0.15)):
        cax = fig.add_axes([0.92, bottom, 0.015, 0.3])
        cb = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=mpl.colors.Normalize(*limits[col]))
        cb.set_label(f"{label} (state mean)", fontsize=6)
        cb.ax.tick_params(labelsize=5)

    plt.subplots_adjust(wspace=0.3, right=0.9)
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

In [ ]:
plt.rcParams.update({"font.size": 6, "axes.titlesize": 7})

plot_paga_grid(adata_mal, save_path=f"{figures_output_dir}/05_paga_malignant.pdf", figsize=(6, 2.2))
plot_paga_grid(adata_mal, group_col="recist_v3", groups=treatment_order,
               save_path=f"{figures_output_dir}/06_paga_malignant_by_recist.pdf", figsize=(7.5, 6))

adata_resp = adata_mal[adata_mal.obs["response_group"].notna()
                       & ~adata_mal.obs["recist_v3"].isin(response_exclude_recist)].copy()
plot_paga_grid(adata_resp, group_col="response_group", groups=response_order,
               save_path=f"{figures_output_dir}/07_paga_malignant_by_response.pdf", figsize=(10.5, 8))

plt.rcParams.update(mpl.rcParamsDefault)